In [0]:
from pyspark.sql.functions import *

spark.conf.set(
    "spark.sql.session.timeZone",
    "UTC"
)

catalog = "crypto_lakehouse"
silver_schema = "silver"
gold_schema = "gold"

spark.sql(
    "CREATE SCHEMA IF NOT EXISTS crypto_lakehouse.gold"
)

display(
    spark.sql(
        "SHOW TABLES IN crypto_lakehouse.gold"
    )
)

database,tableName,isTemporary
gold,dim_asset,false
gold,dim_date,false
gold,dim_exchange,false
gold,dim_wallet,false
gold,fact_asset_movements,false
gold,fact_balance_snapshots,false
gold,fact_daily_balances,false
gold,fact_daily_portfolio,false
gold,fact_daily_prices,false
gold,fact_p2p_transactions,false


####Dim assets

In [0]:
asset_source_df = spark.sql(
    f"""
    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_funding_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_trades

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_p2p_transactions

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_deposits

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_withdrawals

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_transfers

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.binance_price_history

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.giottus_trades

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_account_balances
    WHERE total_balance > 0

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_trades

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_fiat_deposits

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_fiat_withdrawals

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_crypto_deposits

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.coindcx_crypto_withdrawals

    UNION

    SELECT UPPER(asset_spent) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_trades

    UNION

    SELECT UPPER(asset_received) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_trades

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_deposits

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_withdrawals

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_earn_history

    UNION

    SELECT UPPER(asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_usdt_fund_transactions

    UNION

    SELECT UPPER(fiat_asset) AS asset_symbol
    FROM {catalog}.{silver_schema}.mudrex_usdt_fund_transactions
    """
)

asset_source_df = (
    asset_source_df
    .filter(
        col("asset_symbol").isNotNull()
    )
)

print(
    "Useful assets:",
    asset_source_df.count()
)

display(
    asset_source_df.orderBy(
        "asset_symbol"
    )
)

Useful assets: 8


asset_symbol
BTC
ETH
INR
LINK
ONDO
SOL
SUI
USDT


In [0]:
dim_asset_df = (
    asset_source_df
    .select(
        col("asset_symbol")
            .alias("asset_key"),
        col("asset_symbol")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print(
    "Dimension asset rows:",
    dim_asset_df.count()
)

display(
    dim_asset_df.orderBy(
        col("asset_symbol")
    )
)

Dimension asset rows: 8


asset_key,asset_symbol,gold_processed_at
BTC,BTC,2026-10-02T14:52:00.660Z
ETH,ETH,2026-10-02T14:52:00.660Z
INR,INR,2026-10-02T14:52:00.660Z
LINK,LINK,2026-10-02T14:52:00.660Z
ONDO,ONDO,2026-10-02T14:52:00.660Z
SOL,SOL,2026-10-02T14:52:00.660Z
SUI,SUI,2026-10-02T14:52:00.660Z
USDT,USDT,2026-10-02T14:52:00.660Z


In [0]:
(
    dim_asset_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_asset"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.dim_asset"
    )
)

asset_key,asset_symbol,gold_processed_at
ETH,ETH,2026-10-02T14:52:06.516Z
USDT,USDT,2026-10-02T14:52:06.516Z
SUI,SUI,2026-10-02T14:52:06.516Z
SOL,SOL,2026-10-02T14:52:06.516Z
BTC,BTC,2026-10-02T14:52:06.516Z
LINK,LINK,2026-10-02T14:52:06.516Z
ONDO,ONDO,2026-10-02T14:52:06.516Z
INR,INR,2026-10-02T14:52:06.516Z


#### Dim exchange

In [0]:
binance_exchanges_df = (
    spark.table(f"{catalog}.{silver_schema}.binance_account_snapshots")
    .select("exchange")
)

giottus_exchanges_df = (
    spark.table(f"{catalog}.{silver_schema}.giottus_account_balances")
    .select("exchange")
)

coindcx_exchanges_df = (
    spark.table(f"{catalog}.{silver_schema}.coindcx_account_balances")
    .select("exchange")
)

mudrex_exchanges_df = (
    spark.table(
        f"{catalog}.{silver_schema}.mudrex_trades")
    .select("exchange")
)

exchange_source_df = (
    binance_exchanges_df
    .unionByName(
        giottus_exchanges_df
    )
    .unionByName(
        coindcx_exchanges_df
    )
    .unionByName(
        mudrex_exchanges_df
    )
)


dim_exchange_df = (
    exchange_source_df
    .select(
        upper(col("exchange")).alias("exchange_key"),
        col("exchange").alias("exchange_name")
    )
    .filter(col("exchange_key").isNotNull())
    .dropDuplicates(["exchange_key"])
    .withColumn("exchange_type", lit("Centralized Exchange"))
    .withColumn("gold_processed_at", current_timestamp())
)

display(dim_exchange_df)

exchange_key,exchange_name,exchange_type,gold_processed_at
BINANCE,Binance,Centralized Exchange,2026-10-02T14:52:11.491Z
GIOTTUS,Giottus,Centralized Exchange,2026-10-02T14:52:11.491Z
COINDCX,CoinDCX,Centralized Exchange,2026-10-02T14:52:11.491Z
MUDREX,Mudrex,Centralized Exchange,2026-10-02T14:52:11.491Z


In [0]:
(
    dim_exchange_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_exchange"
    )
)

display(
    spark.table(
        f"{catalog}.{gold_schema}.dim_exchange"
    )
)


exchange_key,exchange_name,exchange_type,gold_processed_at
BINANCE,Binance,Centralized Exchange,2026-10-02T14:52:14.438Z
GIOTTUS,Giottus,Centralized Exchange,2026-10-02T14:52:14.438Z
COINDCX,CoinDCX,Centralized Exchange,2026-10-02T14:52:14.438Z
MUDREX,Mudrex,Centralized Exchange,2026-10-02T14:52:14.438Z


####Dim date

In [0]:
date_source_df = spark.sql(
    f"""
    SELECT TO_DATE(open_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_price_history

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_trades

    UNION ALL

    SELECT TO_DATE(transaction_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_p2p_transactions

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_deposits

    UNION ALL

    SELECT TO_DATE(
        COALESCE(completed_at, requested_at)
    ) AS data_date
    FROM {catalog}.{silver_schema}.binance_withdrawals

    UNION ALL

    SELECT TO_DATE(transfer_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_transfers

    UNION ALL

    SELECT TO_DATE(
        SUBSTRING(snapshot_id, 9, 8),
        'yyyyMMdd'
    ) AS data_date
    FROM {catalog}.{silver_schema}.binance_account_snapshots

    UNION ALL

    SELECT TO_DATE(snapshot_time) AS data_date
    FROM {catalog}.{silver_schema}.binance_funding_account_snapshots

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_trades

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_fiat_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_fiat_withdrawals

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_crypto_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.giottus_crypto_withdrawals

    UNION ALL

    SELECT TO_DATE(bronze_ingested_at) AS data_date
    FROM {catalog}.{silver_schema}.giottus_account_balances

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_trades

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_fiat_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_fiat_withdrawals

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_crypto_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_crypto_withdrawals

    UNION ALL

    SELECT TO_DATE(bronze_ingested_at) AS data_date
    FROM {catalog}.{silver_schema}.coindcx_account_balances

    UNION ALL

    SELECT TO_DATE(trade_time) AS data_date
    FROM {catalog}.{silver_schema}.mudrex_trades

    UNION ALL

    SELECT TO_DATE(deposit_time) AS data_date
    FROM {catalog}.{silver_schema}.mudrex_deposits

    UNION ALL

    SELECT TO_DATE(withdrawal_time) AS data_date
    FROM {catalog}.{silver_schema}.mudrex_withdrawals

    UNION ALL

    SELECT TO_DATE(earn_time) AS data_date
    FROM {catalog}.{silver_schema}.mudrex_earn_history

    UNION ALL

    SELECT TO_DATE(transaction_time) AS data_date
    FROM {catalog}.{silver_schema}.mudrex_usdt_fund_transactions
    """
)

date_range_df = (
    date_source_df
    .filter(
        col("data_date").isNotNull()
    )
    .agg(
        min("data_date").alias("start_date"),
        max("data_date").alias("end_date")
    )
)

display(date_range_df)

start_date,end_date
2025-05-29,2026-10-02


In [0]:
dim_date_df = (
    date_range_df
    .select(
        explode(
            sequence(
                col("start_date"),
                col("end_date")
            )
        ).alias("full_date")
    )
    .select(
        date_format(
            col("full_date"),
            "yyyyMMdd"
        )
        .cast("integer")
        .alias("date_key"),

        col("full_date"),

        year(
            col("full_date")
        ).alias("year"),

        quarter(
            col("full_date")
        ).alias("quarter"),

        month(
            col("full_date")
        ).alias("month_number"),

        date_format(
            col("full_date"),
            "MMMM"
        ).alias("month_name"),

        dayofmonth(
            col("full_date")
        ).alias("day_of_month"),

        date_format(
            col("full_date"),
            "EEEE"
        ).alias("day_name"),

        weekofyear(
            col("full_date")
        ).alias("week_of_year"),

        when(
            dayofweek(
                col("full_date")
            ).isin(1, 7),
            True
        )
        .otherwise(False)
        .alias("is_weekend")
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)


print(
    "Date dimension rows:",
    dim_date_df.count()
)


display(
    dim_date_df.orderBy(
        col("full_date")
    )
)

Date dimension rows: 492


date_key,full_date,year,quarter,month_number,month_name,day_of_month,day_name,week_of_year,is_weekend,gold_processed_at
20250529,2025-05-29,2025,2,5,May,29,Thursday,22,false,2026-10-02T14:52:29.489Z
20250530,2025-05-30,2025,2,5,May,30,Friday,22,false,2026-10-02T14:52:29.489Z
20250531,2025-05-31,2025,2,5,May,31,Saturday,22,true,2026-10-02T14:52:29.489Z
20250601,2025-06-01,2025,2,6,June,1,Sunday,22,true,2026-10-02T14:52:29.489Z
20250602,2025-06-02,2025,2,6,June,2,Monday,23,false,2026-10-02T14:52:29.489Z
20250603,2025-06-03,2025,2,6,June,3,Tuesday,23,false,2026-10-02T14:52:29.489Z
20250604,2025-06-04,2025,2,6,June,4,Wednesday,23,false,2026-10-02T14:52:29.489Z
20250605,2025-06-05,2025,2,6,June,5,Thursday,23,false,2026-10-02T14:52:29.489Z
20250606,2025-06-06,2025,2,6,June,6,Friday,23,false,2026-10-02T14:52:29.489Z
20250607,2025-06-07,2025,2,6,June,7,Saturday,23,true,2026-10-02T14:52:29.489Z


In [0]:
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_date"
    )
)


saved_dim_date_df = spark.table(
    f"{catalog}.{gold_schema}.dim_date"
)


print(
    "Saved date rows:",
    saved_dim_date_df.count()
)


display(
    saved_dim_date_df
    .agg(
        min("full_date").alias("start_date"),
        max("full_date").alias("end_date")
    )
)

Saved date rows: 492


start_date,end_date
2025-05-29,2026-10-02


####Dim wallet

In [0]:
from pyspark.sql.functions import col, current_timestamp

wallet_rows = [
    (
        "BINANCE_SPOT",
        "Binance Spot Wallet",
        "SPOT",
        "BINANCE",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "BINANCE_FUNDING",
        "Binance Funding Wallet",
        "FUNDING",
        "BINANCE",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "GIOTTUS_SPOT",
        "Giottus Spot Wallet",
        "SPOT",
        "GIOTTUS",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "PHANTOM_SOLANA",
        "Phantom Solana Wallet",
        "HOT_WALLET",
        "PHANTOM",
        "SELF_CUSTODY",
        "SOLANA",
        True
    ),
    (
        "COINDCX_SPOT",
        "Coindcx Spot Wallet",
        "SPOT",
        "COINDCX",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "MUDREX_SPOT",
        "Mudrex Spot Wallet",
        "SPOT",
        "MUDREX",
        "CUSTODIAL",
        None,
        True
    ),
    (
        "MUDREX_EARN",
        "Mudrex Earn Wallet",
        "EARN",
        "MUDREX",
        "CUSTODIAL",
        None,
        True
    )
]

wallet_columns = [
    "wallet_key",
    "wallet_name",
    "wallet_type",
    "provider",
    "custody_type",
    "network",
    "is_active"
]

dim_wallet_df = (
    spark.createDataFrame(wallet_rows, wallet_columns)
    .withColumn("gold_processed_at", current_timestamp())
)

print("Wallet rows:", dim_wallet_df.count())
display(dim_wallet_df.orderBy("wallet_key"))

Wallet rows: 7


wallet_key,wallet_name,wallet_type,provider,custody_type,network,is_active,gold_processed_at
BINANCE_FUNDING,Binance Funding Wallet,FUNDING,BINANCE,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
BINANCE_SPOT,Binance Spot Wallet,SPOT,BINANCE,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
COINDCX_SPOT,Coindcx Spot Wallet,SPOT,COINDCX,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
GIOTTUS_SPOT,Giottus Spot Wallet,SPOT,GIOTTUS,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
MUDREX_EARN,Mudrex Earn Wallet,EARN,MUDREX,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
MUDREX_SPOT,Mudrex Spot Wallet,SPOT,MUDREX,CUSTODIAL,null,true,2026-10-02T14:52:39.941Z
PHANTOM_SOLANA,Phantom Solana Wallet,HOT_WALLET,PHANTOM,SELF_CUSTODY,SOLANA,true,2026-10-02T14:52:39.941Z


In [0]:
(
    dim_wallet_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(
        f"{catalog}.{gold_schema}.dim_wallet"
    )
)